In [1]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [3]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [4]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [5]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [6]:
class ANN(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_features, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [7]:
model = ANN(X_train.shape[1])

criterion = nn.BCEWithLogitsLoss()

def l1_regularization(model):

    l1_penalty = 0

    for name, param in model.named_parameters():

        if "weight" in name:
            l1_penalty += torch.sum(torch.abs(param))

    return l1_penalty

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [10]:
epochs = 100
lambda_l1=0.001

for epoch in range(epochs):
    model.train()
    total_loss=0

    for x_batch,y_batch in train_loader:
        outputs=model(x_batch)
        loss=criterion(outputs,y_batch)
        l1_penalty=l1_regularization(model)
        final_loss=loss+ lambda_l1*l1_penalty
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {avg_loss:.4f}"
        )

Epoch [10/100], Loss: 0.5667
Epoch [20/100], Loss: 0.3969
Epoch [30/100], Loss: 0.2530
Epoch [40/100], Loss: 0.1772
Epoch [50/100], Loss: 0.1369
Epoch [60/100], Loss: 0.1108
Epoch [70/100], Loss: 0.0941
Epoch [80/100], Loss: 0.0818
Epoch [90/100], Loss: 0.0715
Epoch [100/100], Loss: 0.0644


In [11]:
model.eval()

with torch.no_grad():

    test_logits = model(X_test_tensor)

    probabilities = torch.sigmoid(test_logits)

    predictions = (probabilities >= 0.5).float()

accuracy = accuracy_score(
    y_test_tensor.numpy(),
    predictions.numpy()
)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.9824561403508771


In [12]:
total_weights = 0
zero_weights = 0

for name, param in model.named_parameters():

    if "weight" in name:

        total_weights += param.numel()

        zero_weights += (
            param == 0
        ).sum().item()

print("Total weights:", total_weights)
print("Zero weights:", zero_weights)

print(
    "Zero weight percentage:",
    (zero_weights / total_weights) * 100
)

Total weights: 4000
Zero weights: 0
Zero weight percentage: 0.0
